In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def create_reader_survey(columns, reader_count=60, seed=21):
    """Build a synthetic reader survey; every value derives from the seed.

    Special codes: -1 means the reader refused to answer and 999 means
    the branch system did not record the visit count.
    """
    rng = np.random.default_rng(seed)
    age_bands = ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    memberships = ["Basic", "Plus", "Family", "Student"]
    genres = ["Fiction", "Mystery", "Science", "History", "Poetry", "Travel", "Comics"]
    devices = ["None", "Tablet", "E-reader", "Phone"]
    records = []
    for index in range(reader_count):
        age_band = age_bands[rng.integers(0, len(age_bands))]
        membership = memberships[rng.integers(0, len(memberships))]
        base_books = {"16-24": 2.5, "25-34": 2.0, "35-44": 1.8, "45-54": 2.2, "55-64": 3.0, "65+": 3.6}[age_band]
        books = max(0.0, round(rng.normal(base_books, 1.1), 1))
        minutes = int(max(0, rng.normal(25 + books * 9, 12)))
        genre = genres[rng.integers(0, len(genres))]
        audiobook_share = round(float(np.clip(rng.normal(0.25, 0.18), 0, 1)), 2)
        visits = int(rng.poisson(2.5 if membership != "Student" else 4.0))
        satisfaction = int(np.clip(round(rng.normal(3.8, 0.9)), 1, 5))
        device = devices[rng.integers(0, len(devices))]
        if rng.random() < 0.06:
            books = np.nan
        if rng.random() < 0.05:
            minutes = -1
        if rng.random() < 0.04:
            visits = 999
        if rng.random() < 0.05:
            genre = None
        records.append([f"R{1000 + index}", age_band, membership, books, minutes, genre, audiobook_share, visits, satisfaction, device])
    survey = pd.DataFrame(records, columns=columns)
    survey["AGE_BAND"] = pd.Categorical(survey["AGE_BAND"], categories=age_bands, ordered=True)
    return survey


survey_columns = ["READER_ID", "AGE_BAND", "MEMBERSHIP", "BOOKS_PER_MONTH", "MINUTES_PER_DAY", "FAVOURITE_GENRE", "AUDIOBOOK_SHARE", "BRANCH_VISITS", "SATISFACTION", "EBOOK_DEVICE"]
survey_df = create_reader_survey(survey_columns)
display(survey_df.head())
print("Survey shape:", survey_df.shape)

In [ ]:
known_orders = {
    "AGE_BAND": ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "MEMBERSHIP": ["Student", "Basic", "Plus", "Family"],
    "SATISFACTION": [1, 2, 3, 4, 5],
}
special_codes = {"MINUTES_PER_DAY": [-1], "BRANCH_VISITS": [999]}
chart_columns = ["AGE_BAND", "MEMBERSHIP", "BOOKS_PER_MONTH", "MINUTES_PER_DAY", "FAVOURITE_GENRE", "AUDIOBOOK_SHARE", "BRANCH_VISITS", "SATISFACTION"]
genre_counts = survey_df["FAVOURITE_GENRE"].fillna("Missing").value_counts()
membership_counts = survey_df["MEMBERSHIP"].value_counts()
refused_minutes = int((survey_df["MINUTES_PER_DAY"] == -1).sum())
unknown_visits = int((survey_df["BRANCH_VISITS"] == 999).sum())
missing_books = int(survey_df["BOOKS_PER_MONTH"].isna().sum())
print("Genres:", dict(genre_counts))
print("Memberships:", dict(membership_counts))
print(f"Refused minutes: {refused_minutes}, unknown visits: {unknown_visits}, missing books: {missing_books}")

In [ ]:
def tidy_label(value):
    """Turn a raw category value into a readable axis label."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "Missing"
    return str(value).strip().replace("_", " ").title()


def looks_numeric(series):
    """Return True when most non-empty values parse as numbers."""
    values = series.dropna()
    if values.empty:
        return False
    return pd.to_numeric(values, errors="coerce").notna().mean() >= 0.9


def build_axis(frame, column_name, known_orders, special_codes, max_categories=8):
    """Describe one column for a parallel-coordinates chart.

    Numeric columns keep their scale; special codes and missing values
    sit just below the observed range. Categories use a known order when
    one is given, otherwise their frequency order.
    """
    series = frame[column_name]
    special_values = special_codes.get(column_name, [])
    axis_info = {"column": column_name, "kind": None, "positions": None,
                 "tick_values": [], "tick_labels": [], "notes": []}
    if column_name in known_orders or not looks_numeric(series):
        if column_name in known_orders:
            order = list(known_orders[column_name])
        else:
            counts = series.fillna("Missing").astype(str).value_counts()
            order = list(counts.index[:max_categories])
            if len(counts) > max_categories:
                axis_info["notes"].append(f"{len(counts) - max_categories} rare categories grouped as Other")
                order.append("Other")
        raw_values = series.astype(object).where(series.notna(), "Missing")
        if column_name not in known_orders:
            raw_values = raw_values.astype(str).where(raw_values.astype(str).isin(order), "Other")
        if "Missing" in set(raw_values.astype(str)) and "Missing" not in order:
            order.append("Missing")
        position_map = {value: index for index, value in enumerate(order)}
        axis_info["kind"] = "category"
        axis_info["positions"] = raw_values.map(lambda value: position_map.get(value, position_map.get(str(value), np.nan))).astype(float)
        axis_info["tick_values"] = list(range(len(order)))
        axis_info["tick_labels"] = [tidy_label(value) for value in order]
        return axis_info
    numeric = pd.to_numeric(series, errors="coerce")
    special_mask = numeric.isin(special_values)
    missing_mask = numeric.isna()
    valid = numeric[~special_mask & ~missing_mask]
    if valid.empty:
        axis_info["kind"] = "empty"
        axis_info["positions"] = pd.Series(np.nan, index=series.index)
        return axis_info
    low, high = float(valid.min()), float(valid.max())
    span = high - low if high > low else 1.0
    gap = span * 0.12
    positions = numeric.copy().astype(float)
    tick_values = list(np.linspace(low, high, 5))
    tick_labels = [f"{value:.1f}" if span < 20 else f"{value:.0f}" for value in tick_values]
    next_slot = low - gap
    for code in special_values:
        if (numeric == code).any():
            positions[numeric == code] = next_slot
            tick_values.insert(0, next_slot)
            tick_labels.insert(0, f"code {code}")
            axis_info["notes"].append(f"{int((numeric == code).sum())} values coded {code}")
            next_slot -= gap
    if missing_mask.any():
        positions[missing_mask] = next_slot
        tick_values.insert(0, next_slot)
        tick_labels.insert(0, "Missing")
        axis_info["notes"].append(f"{int(missing_mask.sum())} missing values")
    axis_info.update(kind="numeric", positions=positions, tick_values=tick_values, tick_labels=tick_labels)
    return axis_info

In [ ]:
def parallel_chart_summary(frame, columns, known_orders, special_codes, title_text="Reader survey"):
    """Draw the chart off screen and summarize what it contains."""
    axes_info = [build_axis(frame, column_name, known_orders, special_codes) for column_name in columns]
    figure, axis = plt.subplots(figsize=(11, 4))
    lines_drawn = 0
    scaled = []
    for axis_info in axes_info:
        tick_values = axis_info["tick_values"] or [0.0]
        low, high = min(tick_values), max(tick_values)
        span = high - low if high > low else 1.0
        scaled.append(((axis_info["positions"] - low) / span).to_numpy())
    stacked = np.vstack(scaled)
    for column_index in range(stacked.shape[1]):
        if not np.isnan(stacked[:, column_index]).all():
            axis.plot(range(len(columns)), stacked[:, column_index], alpha=0.4, linewidth=1)
            lines_drawn += 1
    axis.set_xticks(range(len(columns)))
    axis.set_xticklabels([tidy_label(column_name) for column_name in columns], rotation=20)
    axis.set_title(title_text)
    plt.close(figure)
    return {"axes": len(axes_info), "lines": lines_drawn,
            "notes": {axis_info["column"]: axis_info["notes"] for axis_info in axes_info if axis_info["notes"]}}

In [ ]:
overview = parallel_chart_summary(survey_df, chart_columns, known_orders, special_codes)
print("Overview:", overview["axes"], "axes,", overview["lines"], "readers")
for column_name, column_notes in overview["notes"].items():
    print(f"  {column_name}: {'; '.join(column_notes)}")
axis_kinds = [build_axis(survey_df, column_name, known_orders, special_codes)["kind"] for column_name in chart_columns]
print("Axis kinds:", dict(zip(chart_columns, axis_kinds)))

In [ ]:
by_age_df = survey_df.groupby("AGE_BAND", observed=True).agg(
    readers=("READER_ID", "count"),
    books=("BOOKS_PER_MONTH", "mean"),
    satisfaction=("SATISFACTION", "mean"),
).round(2)
print(by_age_df)

In [ ]:
def add_survey_column(frame, column_name, values, position=None):
    """
    Add a new survey column without changing the original frame.

    Parameters
    ----------
    frame : pandas DataFrame
        Survey table.
    column_name : str
        Name of the new column.
    values : list, numpy array, pandas Series or scalar
        One value per reader; a scalar is repeated for every reader.
    position : int or None
        Where to insert the column; None appends it at the end.
    """
    updated = frame.copy()
    if np.isscalar(values):
        values = [values] * len(updated)
    values = list(values)
    if len(values) != len(updated):
        raise ValueError(
            f"Expected {len(updated)} values for '{column_name}', "
            f"received {len(values)}."
        )
    if position is None or position >= len(updated.columns):
        updated[column_name] = values
    else:
        updated.insert(position, column_name, values)
    return updated

In [ ]:
extra_rng = np.random.default_rng(77)
podcast_hours = extra_rng.choice([np.nan, 0.0, 0.5, 1.0, 2.0, 3.5], size=len(survey_df), p=[0.08, 0.3, 0.2, 0.2, 0.15, 0.07])
survey_with_podcasts_df = add_survey_column(survey_df, "PODCAST_HOURS", podcast_hours)
display(survey_with_podcasts_df.head())

In [ ]:
extended = parallel_chart_summary(survey_with_podcasts_df, chart_columns + ["PODCAST_HOURS"], known_orders, special_codes, "With podcast hours")
print("Extended:", extended["axes"], "axes")
heavy_listeners = [reader for reader, hours in zip(survey_with_podcasts_df["READER_ID"], survey_with_podcasts_df["PODCAST_HOURS"]) if hours == 3.5]
print("Heavy podcast listeners:", len(heavy_listeners))